In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter
from pathlib import Path


import matplotlib.colors as mcolors
import matplotlib.ticker as mticker

# =========================================================
# 0. 路径与参数配置
# =========================================================
REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "results" / "hmof_plotting_data"
PB_CSV = DATA_DIR / "hmof_existing_results.csv"
PP_CSV = DATA_DIR / "hmof_pp_results_8p8t.csv"

OUT_DIR = REPO_ROOT / "docs" / "figures" / "hmof_consistency"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MERGED_PROP_CSV = OUT_DIR / "pb_pp_merged_for_plot.csv"
MERGED_TIME_CSV = OUT_DIR / "merged_time_compare.csv"
LCD_OUTLIER_CSV = OUT_DIR / "lcd_outliers_for_check.csv"

# pyPore 线程数，仅当 CPU time 缺失时用于 fallback
PP_THREADS = 8

# 散点模式:
# "hexbin" or "scatter"
POINT_MODE = "hexbin"

# hexbin 网格数
HEXBIN_GRIDSIZE = 50

# histogram bins
MARGINAL_BINS = 50

# LCD 偏离阈值
LCD_ABS_DIFF_THRESH_A = 1.0
LCD_REL_DIFF_THRESH = 0.15
LCD_TOPN = 100


# =========================================================
# 1. 全局绘图风格：论文可读
# =========================================================
def set_paper_style():
    mpl.rcParams.update({
        "figure.figsize": (7.25, 5.95),
        "figure.dpi": 140,
        "savefig.dpi": 450,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.03,

        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans", "Liberation Sans"],
        "font.size": 24,
        "axes.labelsize": 24,
        "axes.titlesize": 24,
        "xtick.labelsize": 20,
        "ytick.labelsize": 20,
        "legend.fontsize": 20,

        "axes.linewidth": 1.5,
        "xtick.major.width": 0.9,
        "ytick.major.width": 0.9,
        "xtick.minor.width": 0.7,
        "ytick.minor.width": 0.7,
        "xtick.major.size": 4.5,
        "ytick.major.size": 4.5,
        "xtick.minor.size": 2.5,
        "ytick.minor.size": 2.5,

        "axes.grid": False,
        "legend.frameon": False,
        "axes.unicode_minus": False,
        "mathtext.default": "regular",
    })


# =========================================================
# 2. 偏蓝色 colormap
# =========================================================
def make_soft_blue_cmap():
    colors = [
        "#e0e6ec",
        "#c3d0de",
        "#bcd1eb",
        "#9ebadb",
        "#7fa2ca",
        "#628dbb",
        "#4874a3",
        "#39638f",
        "#2A4970",
    ]
    return LinearSegmentedColormap.from_list("soft_blue", colors, N=256)


SOFT_BLUE_CMAP = make_soft_blue_cmap()


# =========================================================
# 3. 基础工具函数
# =========================================================
def to_numeric(series):
    return pd.to_numeric(series, errors="coerce")


def parse_cpu_percent_to_ratio(series):
    s = series.astype(str).str.replace("%", "", regex=False)
    return pd.to_numeric(s, errors="coerce") / 100.0


def build_cpu_time_from_elapsed_and_cpu_percent(elapsed_s, cpu_percent_ratio):
    return elapsed_s * cpu_percent_ratio


def _clean_xy(x, y, positive_only=False):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if positive_only:
        mask &= (x > 0) & (y > 0)
    return x[mask], y[mask], mask


def _compute_xy_metrics(x, y):
    if len(x) < 2:
        return np.nan, np.nan, np.nan
    r = np.corrcoef(x, y)[0, 1]
    me = np.mean(y - x)
    mae = np.mean(np.abs(y - x))
    rmse = np.sqrt(np.mean((y - x) ** 2))
    return r, me,mae, rmse


def _compute_log_ratio_metrics(x, y):
    if len(x) < 2:
        return np.nan, np.nan, np.nan
    lx = np.log10(x)
    ly = np.log10(y)
    r = np.corrcoef(lx, ly)[0, 1]
    mae_ratio = np.mean(np.abs(np.log10(y / x)))
    rmse_ratio = np.sqrt(np.mean((np.log10(y / x)) ** 2))
    return r, mae_ratio, rmse_ratio


def _get_equal_axis_limits_linear(x, y, pad_ratio=0.03):
    vmin = min(np.min(x), np.min(y))
    vmax = max(np.max(x), np.max(y))
    if vmin == vmax:
        pad = 0.05 * abs(vmin) if vmin != 0 else 1.0
    else:
        pad = pad_ratio * (vmax - vmin)
    return vmin - pad, vmax + pad


def _get_equal_axis_limits_log10(lx, ly, pad_ratio=0.03):
    vmin = min(np.min(lx), np.min(ly))
    vmax = max(np.max(lx), np.max(ly))
    if vmin == vmax:
        vmin -= 0.5
        vmax += 0.5
    else:
        pad = pad_ratio * (vmax - vmin)
        vmin -= pad
        vmax += pad
    return vmin, vmax


# def _log_tick_formatter(val, pos=None):
#     real = 10 ** val
#     if real >= 100:
#         return f"{real:.0f}"
#     elif real >= 10:
#         return f"{real:.1f}"
#     elif real >= 1:
#         return f"{real:.2f}"
#     else:
#         return f"{real:.2g}"


def _style_colorbar(cb, label="log10(count)"):
    cb.set_label(label)
    cb.outline.set_linewidth(0.8)
    cb.ax.tick_params(labelsize=20, width=0.8, length=3.5)


# =========================================================
# 4. 线性图：带边缘分布图 + colorbar 在 hist_y 右边
# =========================================================
def scatter_with_marginal(
    x,
    y,
    xlabel,
    ylabel,
    out_png=None,
    title=None,
    bins=35,
    point_mode="hexbin",
    gridsize=45,
    cmap=SOFT_BLUE_CMAP,
):
    x, y, _ = _clean_xy(x, y, positive_only=False)
    if len(x) == 0:
        print(f"[skip] {title}: no valid data")
        return None, None

    r, me, mae, rmse = _compute_xy_metrics(x, y)
    vmin, vmax = _get_equal_axis_limits_linear(x, y)

    fig = plt.figure(figsize=(7.25, 5.95))
    gs = fig.add_gridspec(
        2, 3,
        width_ratios=(5.45, 0.42, 0.18),
        height_ratios=(0.42, 5.45),
        wspace=0.02,
        hspace=0.02,
    )

    ax_histx = fig.add_subplot(gs[0, 0])
    ax_scatter = fig.add_subplot(gs[1, 0])
    ax_histy = fig.add_subplot(gs[1, 1], sharey=ax_scatter)
    ax_cbar = fig.add_subplot(gs[1, 2])

    # ax_scatter.set_box_aspect(1)

    if point_mode == "hexbin":
        hb = ax_scatter.hexbin(
            x, y,
            gridsize=gridsize,
            mincnt=10,
            # bins="log",
            linewidths=0.0,
            cmap=cmap,
            zorder=2,
        )
        cb = plt.colorbar(hb, cax=ax_cbar)
        _style_colorbar(cb, label="Counts")
    else:
        ax_cbar.axis("off")
        ax_scatter.scatter(
            x, y,
            s=16,
            alpha=0.68,
            c="#5f8fbd",
            edgecolors="none",
            zorder=2,
        )

    ax_scatter.plot(
        [vmin, vmax], [vmin, vmax],
        linestyle="--",
        linewidth=1.5,
        color="#666666",
        zorder=10,
    )

    ax_scatter.set_xlim(vmin, vmax)
    ax_scatter.set_ylim(vmin, vmax)
    ax_scatter.set_xlabel(xlabel)
    ax_scatter.set_ylabel(ylabel)

    txt = f"N = {len(x)}\nR = {r:.4f}\nME = {me:.4g}\nRMSE = {rmse:.4g}"
    ax_scatter.text(
        0.03, 0.97, txt,
        transform=ax_scatter.transAxes,
        ha="left", va="top",
        fontsize=20,
        bbox=dict(boxstyle="round,pad=0.28", fc="white", ec="0.82", alpha=0.92),
        zorder=20,
    )
    ax_histx.hist(
        x,
        bins=bins,
        color="#bed0e6",
        edgecolor="none",
        alpha=0.98,
    )
    ax_histx.set_xlim(vmin, vmax)
    ax_histx.tick_params(axis="x", labelbottom=False, bottom=False)
    ax_histx.tick_params(axis="y", labelleft=False, left=False)

    ax_histy.hist(
        y,
        bins=bins,
        orientation="horizontal",
        color="#bed0e6",
        edgecolor="none",
        alpha=0.98,
    )
    ax_histy.set_ylim(vmin, vmax)
    ax_histy.tick_params(axis="y", labelleft=False, left=False)
    ax_histy.tick_params(axis="x", labelbottom=False, bottom=False)

    for ax in (ax_histx, ax_histy):
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.set_facecolor("none")

    # if title:
        # fig.suptitle(title, y=0.995, fontsize=18)

    # plt.tight_layout()
    if out_png is not None:
        fig.savefig(out_png)
        plt.close(fig)

    return fig, (ax_scatter, ax_histx, ax_histy, ax_cbar)


# =========================================================
# 5. 对数图：先 log10，再 hexbin
# =========================================================
def scatter_with_marginal_log(
    x,
    y,
    xlabel,
    ylabel,
    out_png=None,
    title=None,
    bins=35,
    point_mode="hexbin",
    gridsize=45,
    cmap=SOFT_BLUE_CMAP,
):
    x, y, _ = _clean_xy(x, y, positive_only=True)
    if len(x) == 0:
        print(f"[skip] {title}: no valid positive data")
        return None, None

    r, mae_ratio, rmse_ratio = _compute_log_ratio_metrics(x, y)

    lx = np.log10(x)
    ly = np.log10(y)
    vlo, vhi = _get_equal_axis_limits_log10(lx, ly)

    fig = plt.figure(figsize=(7.25, 5.95))
    gs = fig.add_gridspec(
        2, 3,
        width_ratios=(5.45, 0.42, 0.18),
        height_ratios=(0.42, 5.45),
        wspace=0.02,
        hspace=0.02,
    )

    ax_histx = fig.add_subplot(gs[0, 0])
    ax_scatter = fig.add_subplot(gs[1, 0])
    ax_histy = fig.add_subplot(gs[1, 1], sharey=ax_scatter)
    ax_cbar = fig.add_subplot(gs[1, 2])

    # ax_scatter.set_box_aspect(1)

    if point_mode == "hexbin":
        hb = ax_scatter.hexbin(
            lx, ly,
            gridsize=gridsize,
            mincnt=1,
            # bins="log",
            extent=(vlo, vhi, vlo, vhi),
            linewidths=0.0,
            cmap=cmap,
            zorder=2,
        )
        cb = plt.colorbar(hb, cax=ax_cbar)
        _style_colorbar(cb, label="count")
    else:
        ax_cbar.axis("off")
        ax_scatter.scatter(
            lx, ly,
            s=16,
            alpha=0.68,
            c="#5f8fbd",
            edgecolors="none",
            zorder=2,
        )

    ax_scatter.plot(
        [vlo, vhi], [vlo, vhi],
        linestyle="--",
        linewidth=1.5,
        color="#666666",
        zorder=10,
    )

    # ax_scatter.set_xlim(vlo, vhi)
    # ax_scatter.set_ylim(vlo, vhi)
    # # ax_scatter.set_xscale('log')
    # # ax_scatter.set_yscale('log')
    # ax_scatter.set_xlabel(xlabel)
    # ax_scatter.set_ylabel(ylabel)

    ax_scatter.set_xlim(vlo, vhi)
    ax_scatter.set_ylim(vlo, vhi)
    ax_scatter.set_xlabel(xlabel)
    ax_scatter.set_ylabel(ylabel)

    ticks = np.arange(int(np.ceil(vlo)), int(np.floor(vhi)) + 1, 1)
    ax_scatter.xaxis.set_major_locator(mticker.FixedLocator(ticks))
    ax_scatter.yaxis.set_major_locator(mticker.FixedLocator(ticks))
    ax_scatter.xaxis.set_major_formatter(
        mticker.FuncFormatter(lambda v, p: rf"$10^{{{int(round(v))}}}$")
    )
    ax_scatter.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda v, p: rf"$10^{{{int(round(v))}}}$")
    )
    ax_scatter.xaxis.set_minor_locator(mticker.NullLocator())
    ax_scatter.yaxis.set_minor_locator(mticker.NullLocator())


    # txt = (
    #     f"N = {len(x)}\n"
    #     f"corr(log10) = {r:.4f}\n"
    #     f"mean |log10(y/x)| = {mae_ratio:.4f}\n"
    #     f"rmse log10(y/x) = {rmse_ratio:.4f}"
    # )
    # ax_scatter.text(
    #     0.03, 0.97, txt,
    #     transform=ax_scatter.transAxes,
    #     ha="left", va="top",
    #     fontsize=10,
    #     bbox=dict(boxstyle="round,pad=0.28", fc="white", ec="0.82", alpha=0.92),
    #     zorder=20,
    # )

    ax_histx.hist(
        lx,
        bins=bins,
        color="#bed0e6",
        edgecolor="none",
        alpha=0.98,
    )
    ax_histx.set_xlim(vlo, vhi)
    ax_histx.tick_params(axis="x", labelbottom=False, bottom=False)
    ax_histx.tick_params(axis="y", labelleft=False, left=False)

    ax_histy.hist(
        ly,
        bins=bins,
        orientation="horizontal",
        color="#bed0e6",
        edgecolor="none",
        alpha=0.98,
    )
    ax_histy.set_ylim(vlo, vhi)
    ax_histy.tick_params(axis="y", labelleft=False, left=False)
    ax_histy.tick_params(axis="x", labelbottom=False, bottom=False)

    for ax in (ax_histx, ax_histy):
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.set_facecolor("none")

    if title:
        fig.suptitle(title, y=0.995, fontsize=24)

    if out_png is not None:
        fig.savefig(out_png)
        plt.close(fig)

    return fig, (ax_scatter, ax_histx, ax_histy, ax_cbar)


# =========================================================
# 6. 其他图
# =========================================================
def plot_speedup_hist(speedup, out_png=None, bins=40):
    s = np.asarray(speedup, dtype=float)
    s = s[np.isfinite(s) & (s > 0)]
    if len(s) == 0:
        print("[skip] speedup hist: no valid data")
        return None, None

    median = np.median(s)
    mean = np.mean(s)

    fig, ax = plt.subplots(figsize=(6.9, 5.2))
    ax.hist(
        s,
        bins=np.logspace(np.log10(s.min()), np.log10(s.max()), bins),
        color="#9fb8d8",
        edgecolor="none",
        alpha=0.98,
    )
    ax.set_xscale("log")
    ax.set_xlabel("Speedup = PB wall time / PxPore wall time")
    ax.set_ylabel("Count")
    ax.set_title("Speedup Distribution")

    txt = f"N = {len(s)}\nmean = {mean:.3f}\nmedian = {median:.3f}"
    ax.text(
        0.97, 0.97, txt,
        transform=ax.transAxes,
        ha="right", va="top",
        fontsize=20,
        bbox=dict(boxstyle="round,pad=0.28", fc="white", ec="0.82", alpha=0.92),
    )

    if out_png is not None:
        fig.savefig(out_png)
        plt.close(fig)

    return fig, ax

def plot_speedup_vs_size(
    x,
    y,
    xlabel,
    ylabel,
    out_png=None,
    title=None,
    bins=35,
    point_mode="hexbin",
    gridsize=45,
    cmap=SOFT_BLUE_CMAP,
):
    x, y, _ = _clean_xy(x, y, positive_only=True)
    if len(x) == 0:
        print(f"[skip] {title}: no valid positive data")
        return None, None

    r, mae_ratio, rmse_ratio = _compute_log_ratio_metrics(x, y)

    lx = np.log10(x)
    ly = np.log10(y)
    # vlo, vhi = _get_equal_axis_limits_log10(lx, ly)

    fig = plt.figure(figsize=(7.25, 5.95))
    gs = fig.add_gridspec(
        2, 3,
        width_ratios=(5.45, 0.42, 0.18),
        height_ratios=(0.42, 5.45),
        wspace=0.02,
        hspace=0.02,
    )

    ax_histx = fig.add_subplot(gs[0, 0])
    ax_scatter = fig.add_subplot(gs[1, 0])
    ax_histy = fig.add_subplot(gs[1, 1], sharey=ax_scatter)
    ax_cbar = fig.add_subplot(gs[1, 2])

    # ax_scatter.set_box_aspect(1)

    if point_mode == "hexbin":
        hb = ax_scatter.hexbin(
            lx, ly,
            gridsize=gridsize,
            mincnt=1,
            # bins="log",
            extent=(lx.min(), lx.max(), ly.min(), ly.max()),
            linewidths=0.0,
            cmap=cmap,
            zorder=2,
        )
        cb = plt.colorbar(hb, cax=ax_cbar)
        _style_colorbar(cb, label="count")
    else:
        ax_cbar.axis("off")
        ax_scatter.scatter(
            lx, ly,
            s=16,
            alpha=0.68,
            c="#5f8fbd",
            edgecolors="none",
            zorder=2,
        )

    # ax_scatter.plot(
    #     [vlo, vhi], [vlo, vhi],
    #     linestyle="--",
    #     linewidth=1.5,
    #     color="#666666",
    #     zorder=10,
    # )


    # ax_scatter.set_xlim(vlo, vhi)
    # ax_scatter.set_ylim(vlo, vhi)

    ax_scatter.set_xlabel(xlabel)
    ax_scatter.set_ylabel(ylabel)
    ax_scatter.axhline(
        np.log10(64),
        color="red",
        linestyle="--",
        linewidth=1.2,
        alpha=0.8,
        label="64 threads",
    )
    ax_scatter.legend(loc="upper left", frameon=False)
    xticks = np.arange(int(np.ceil(lx.min())), int(np.floor(lx.max())) + 1, 1)
    yticks = np.arange(int(np.ceil(ly.min())), int(np.floor(ly.max())) + 1, 1)
    ax_scatter.xaxis.set_major_locator(mticker.FixedLocator(xticks))
    ax_scatter.yaxis.set_major_locator(mticker.FixedLocator(yticks))
    ax_scatter.xaxis.set_major_formatter(
        mticker.FuncFormatter(lambda v, p: rf"$10^{{{int(round(v))}}}$")
    )
    ax_scatter.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda v, p: rf"$10^{{{int(round(v))}}}$")
    )
    ax_scatter.xaxis.set_minor_locator(mticker.NullLocator())
    ax_scatter.yaxis.set_minor_locator(mticker.NullLocator())


    # txt = (
    #     f"N = {len(x)}\n"
    #     f"corr(log10) = {r:.4f}\n"
    #     f"mean |log10(y/x)| = {mae_ratio:.4f}\n"
    #     f"rmse log10(y/x) = {rmse_ratio:.4f}"
    # )
    # ax_scatter.text(
    #     0.03, 0.97, txt,
    #     transform=ax_scatter.transAxes,
    #     ha="left", va="top",
    #     fontsize=10,
    #     bbox=dict(boxstyle="round,pad=0.28", fc="white", ec="0.82", alpha=0.92),
    #     zorder=20,
    # )

    ax_histx.hist(
        lx,
        bins=bins,
        color="#bed0e6",
        edgecolor="none",
        alpha=0.98,
    )
    ax_histx.set_xlim(lx.min(), lx.max())
    ax_histx.tick_params(axis="x", labelbottom=False, bottom=False)
    ax_histx.tick_params(axis="y", labelleft=False, left=False)

    ax_histy.hist(
        ly,
        bins=bins,
        orientation="horizontal",
        color="#bed0e6",
        edgecolor="none",
        alpha=0.98,
    )
    ax_histy.set_ylim(ly.min(), ly.max())
    ax_histy.tick_params(axis="y", labelleft=False, left=False)
    ax_histy.tick_params(axis="x", labelbottom=False, bottom=False)

    for ax in (ax_histx, ax_histy):
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.set_facecolor("none")

    if title:
        fig.suptitle(title, y=0.995, fontsize=24)

    if out_png is not None:
        fig.savefig(out_png)
        plt.close(fig)

    return fig, (ax_scatter, ax_histx, ax_histy, ax_cbar)
# def plot_speedup_vs_size(
#     size,
#     speedup,
#     xlabel,
#     out_png=None,
#     point_mode="hexbin",
#     gridsize=45,
#     cmap=SOFT_BLUE_CMAP,
# ):
#     x = np.asarray(size, dtype=float)
#     y = np.asarray(speedup, dtype=float)

#     mask = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
#     x = x[mask]
#     y = y[mask]

#     if len(x) == 0:
#         print("[skip] speedup vs size: no valid data")
#         return None, None

#     lx = np.log10(x)
#     ly = np.log10(y)

#     fig, ax = plt.subplots(figsize=(6.9, 5.2))
#     ax.set_box_aspect(1)

#     if point_mode == "hexbin":
#         hb = ax.hexbin(
#             lx, ly,
#             # x,y,
#             gridsize=gridsize,
#             mincnt=1,
#             extent=(lx.min(), lx.max(), ly.min(), ly.max()),
#             # bins="log",
#             linewidths=0.0,
#             cmap=cmap,
#         )
#         cb = plt.colorbar(hb, ax=ax)
#         _style_colorbar(cb, label="count")
#     else:
#         ax.scatter(
#             lx, ly,
#             s=16,
#             alpha=0.68,
#             c="#5f8fbd",
#             edgecolors="none",
#         )

#     ax.axhline(
#         np.log10(64),
#         color="red",
#         linestyle="--",
#         linewidth=1.2,
#         alpha=0.8,
#         label="64 threads speedup threshold",
#     )
#     ax.legend(loc="upper left", frameon=False)

#     ax.xaxis.set_major_locator(mticker.FixedLocator(np.arange(int(np.ceil(lx.min())), int(np.floor(lx.max())) + 1, 1)))
#     ax.yaxis.set_major_locator(mticker.FixedLocator(np.arange(int(np.ceil(ly.min())), int(np.floor(ly.max())) + 1, 1)))
#     ax.xaxis.set_major_formatter(
#         mticker.FuncFormatter(lambda v, p: rf"$10^{{{int(round(v))}}}$")
#     )
#     ax.yaxis.set_major_formatter(
#         mticker.FuncFormatter(lambda v, p: rf"$10^{{{int(round(v))}}}$")
#     )
#     ax.xaxis.set_minor_locator(mticker.NullLocator())
#     ax.yaxis.set_minor_locator(mticker.NullLocator())

#     ax.set_xlabel(xlabel)
#     ax.set_ylabel("Speedup = PB wall / pyPore wall")
#     ax.set_title("Speedup vs System Size")

#     if out_png is not None:
#         fig.savefig(out_png)
#         plt.close(fig)

#     return fig, ax


def plot_time_vs_size(
    pb_size,
    pb_time,
    pp_size,
    pp_time,
    xlabel,
    out_png=None,
    time_kind="wall",
):
    x1 = np.asarray(pb_size, dtype=float)
    y1 = np.asarray(pb_time, dtype=float)
    x2 = np.asarray(pp_size, dtype=float)
    y2 = np.asarray(pp_time, dtype=float)

    m1 = np.isfinite(x1) & np.isfinite(y1) & (x1 > 0) & (y1 > 0)
    m2 = np.isfinite(x2) & np.isfinite(y2) & (x2 > 0) & (y2 > 0)

    fig, ax = plt.subplots(figsize=(6.9, 5.2))

    if np.any(m1):
        ax.scatter(
            x1[m1], y1[m1],
            s=16, alpha=0.58, c="#7898bd", edgecolors="none", label="PoreBlazer"
        )
    if np.any(m2):
        ax.scatter(
            x2[m2], y2[m2],
            s=16, alpha=0.78, c="#315682", edgecolors="none", label="PxPore"
        )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(f"{'Wall' if time_kind == 'wall' else 'CPU'} time (s)")
    ax.set_title(f"{'Wall' if time_kind == 'wall' else 'CPU'} Time vs System Size")
    ax.legend()

    if out_png is not None:
        fig.savefig(out_png)
        plt.close(fig)

    return fig, ax




# =========================================================
# 8. 读入与合并
# =========================================================
set_paper_style()




# =========================================================
# 14. 输出说明
# =========================================================
print(f"all outputs saved in: {OUT_DIR}")
print("generated files may include:")
print("  - pb_pp_merged_for_plot.csv")
print("  - merged_time_compare.csv")
print("  - *_surface_area*.png")
print("  - *_pore_volume*.png")
print("  - pld_A.png / lcd_A.png / lcd_global_A.png / lcd_global_vs_local_nm.png")
print("  - wall_time_pb_vs_pp.png / cpu_time_pb_vs_pp.png")
print("  - speedup_hist.png / speedup_vs_voxels.png")
print("  - wall_time_vs_voxels.png / cpu_time_vs_voxels.png")
print("  - lcd_outliers_for_check.csv")

In [ ]:
pb = pd.read_csv(PB_CSV)
pp = pd.read_csv(PP_CSV)

# 统一旧 1p64t (_numa) 与新 8p8t (_8p8t_2cpu) 的结构键
pp["name_raw"] = pp["name"].astype(str)
pp["name"] = pp["name_raw"].str.replace(
    r"(?:_numa|_8p8t(?:_\d+cpu)?)$", "", regex=True
)

# 只保留成功项
if "status" in pp.columns:
    pp = pp[pp["status"] == "ok"].copy()

if pb["name"].duplicated().any():
    raise ValueError("PB_CSV contains duplicate names")
if pp["name"].duplicated().any():
    raise ValueError("PP_CSV contains duplicate canonical names after suffix cleanup")

df = pb.merge(
    pp, on="name", suffixes=("_pb", "_pp"), validate="one_to_one"
)
pb_only_names = sorted(set(pb["name"]) - set(pp["name"]))
pp_only_names = sorted(set(pp["name"]) - set(pb["name"]))
if pp_only_names or len(df) != len(pp):
    raise ValueError(
        f"Successful PP rows not matched to PB: {len(pp_only_names)}"
    )
if pb_only_names:
    unmatched_path = OUT_DIR / "pb_without_successful_pp.csv"
    pd.DataFrame({"name": pb_only_names}).to_csv(unmatched_path, index=False)
    print(f"[warn] PB rows without successful PP result: {len(pb_only_names)}")
    print(f"saved: {unmatched_path}")
print(f"merged rows = {len(df)}")


# =========================================================
# 9. 单位统一与派生列
# =========================================================
# PB surface: A^2 -> nm^2
if "total_S_AC_A2" in df.columns:
    df["pb_total_S_nm2"] = df["total_S_AC_A2"] / 100.0
if "net_S_AC_A2" in df.columns:
    df["pb_net_S_nm2"] = df["net_S_AC_A2"] / 100.0

# PB volume: A^3 -> nm^3
if "total_V_PO_A3" in df.columns:
    df["pb_total_V_nm3"] = df["total_V_PO_A3"] / 1000.0
if "net_V_PO_A3" in df.columns:
    df["pb_net_V_nm3"] = df["net_V_PO_A3"] / 1000.0

# pyPore pore size: nm -> A
if "PLD_nm" in df.columns:
    df["pp_PLD_A"] = df["PLD_nm"] * 10.0
if "LCD_nm" in df.columns:
    df["pp_LCD_A"] = df["LCD_nm"] * 10.0
if "LCD_global_nm" in df.columns:
    df["pp_LCD_global_A"] = df["LCD_global_nm"] * 10.0

df.to_csv(MERGED_PROP_CSV, index=False)
print(f"saved: {MERGED_PROP_CSV}")


# =========================================================
# 10. 时间列整理
# =========================================================
# PB
if "time_elapsed_seconds_pb" in df.columns:
    df["pb_wall_s"] = to_numeric(df["time_elapsed_seconds_pb"])
if "time_user_seconds_pb" in df.columns:
    df["pb_user_s"] = to_numeric(df["time_user_seconds_pb"])
if "time_sys_seconds_pb" in df.columns:
    df["pb_sys_s"] = to_numeric(df["time_sys_seconds_pb"])
if "time_cpu_percent_pb" in df.columns:
    df["pb_cpu_percent_ratio"] = parse_cpu_percent_to_ratio(df["time_cpu_percent_pb"])

if ("pb_user_s" in df.columns) and ("pb_sys_s" in df.columns):
    df["pb_cpu_s_from_usrsys"] = df["pb_user_s"] + df["pb_sys_s"]

if ("pb_wall_s" in df.columns) and ("pb_cpu_percent_ratio" in df.columns):
    df["pb_cpu_s_from_percent"] = build_cpu_time_from_elapsed_and_cpu_percent(
        df["pb_wall_s"], df["pb_cpu_percent_ratio"]
    )

if ("pb_cpu_s_from_usrsys" in df.columns) or ("pb_cpu_s_from_percent" in df.columns):
    df["pb_cpu_s"] = df.get("pb_cpu_s_from_usrsys", pd.Series(np.nan, index=df.index)).where(
        df.get("pb_cpu_s_from_usrsys", pd.Series(np.nan, index=df.index)).notna(),
        df.get("pb_cpu_s_from_percent", pd.Series(np.nan, index=df.index))
    )

# pyPore
if "time_elapsed_seconds_pp" in df.columns:
    df["pp_wall_s"] = to_numeric(df["time_elapsed_seconds_pp"])
if "time_user_seconds_pp" in df.columns:
    df["pp_user_s"] = to_numeric(df["time_user_seconds_pp"])
if "time_sys_seconds_pp" in df.columns:
    df["pp_sys_s"] = to_numeric(df["time_sys_seconds_pp"])
if "time_cpu_percent_pp" in df.columns:
    df["pp_cpu_percent_ratio"] = parse_cpu_percent_to_ratio(df["time_cpu_percent_pp"])

if ("pp_user_s" in df.columns) and ("pp_sys_s" in df.columns):
    df["pp_cpu_s_from_usrsys"] = df["pp_user_s"] + df["pp_sys_s"]

if ("pp_wall_s" in df.columns) and ("pp_cpu_percent_ratio" in df.columns):
    df["pp_cpu_s_from_percent"] = build_cpu_time_from_elapsed_and_cpu_percent(
        df["pp_wall_s"], df["pp_cpu_percent_ratio"]
    )

if ("pp_cpu_s_from_usrsys" in df.columns) or ("pp_cpu_s_from_percent" in df.columns):
    df["pp_cpu_s"] = df.get("pp_cpu_s_from_usrsys", pd.Series(np.nan, index=df.index)).where(
        df.get("pp_cpu_s_from_usrsys", pd.Series(np.nan, index=df.index)).notna(),
        df.get("pp_cpu_s_from_percent", pd.Series(np.nan, index=df.index))
    )

if "pp_wall_s" in df.columns:
    df["pp_cpu_s_proxy_threads"] = df["pp_wall_s"] * PP_THREADS

if "pp_cpu_s" in df.columns:
    df["pp_cpu_s_fallback"] = df["pp_cpu_s"].where(
        df["pp_cpu_s"].notna(),
        df.get("pp_cpu_s_proxy_threads", pd.Series(np.nan, index=df.index))
    )

if ("pb_wall_s" in df.columns) and ("pp_wall_s" in df.columns):
    df["speedup_wall"] = df["pb_wall_s"] / df["pp_wall_s"]

if ("pb_cpu_s" in df.columns) and ("pp_cpu_s_fallback" in df.columns):
    df["cpu_ratio_pb_over_pp"] = df["pb_cpu_s"] / df["pp_cpu_s_fallback"]

# size columns
if "voxels" in df.columns:
    df["size_voxels"] = to_numeric(df["voxels"])
if "atoms" in df.columns:
    df["size_atoms_pp"] = to_numeric(df["atoms"])
if "Vcell_nm3" in df.columns:
    df["size_vcell_nm3"] = to_numeric(df["Vcell_nm3"])

df.to_csv(MERGED_TIME_CSV, index=False)
print(f"saved: {MERGED_TIME_CSV}")

In [ ]:
# =========================================================
# 11. 批量物性作图
# =========================================================
set_paper_style()
pairs = [
    {
        "title": "Total Surface Area",
        "pb_col": "pb_total_S_nm2",
        "pp_col": "Stotal_nm2",
        "xlabel": r"PoreBlazer $S_{total}$ (nm$^2$)",
        "ylabel": r"PxPore $S_{total}$ (nm$^2$)",
        "png": OUT_DIR / "total_surface_area_nm2.png",
        "kind": "linear",
    },
    {
        "title": "Accessible Surface Area",
        "pb_col": "pb_net_S_nm2",
        "pp_col": "Sacc_nm2",
        "xlabel": r"PoreBlazer $S_{acc}$ (nm$^2$)",
        "ylabel": r"PxPore $S_{acc}$ (nm$^2$)",
        "png": OUT_DIR / "accessible_surface_area_nm2.png",
        "kind": "linear",
    },
    {
        "title": "Total Pore Volume",
        "pb_col": "pb_total_V_nm3",
        "pp_col": "Vvoid_nm3",
        "xlabel": r"PoreBlazer $V_{total}$ (nm$^3$)",
        "ylabel": r"PxPore $V_{total}$ (nm$^3$)",
        "png": OUT_DIR / "total_pore_volume_nm3.png",
        "kind": "linear",
    },
    {
        "title": "Accessible Pore Volume",
        "pb_col": "pb_net_V_nm3",
        "pp_col": "Vacc_nm3",
        "xlabel": r"PoreBlazer $V_{acc}$ (nm$^3$)",
        "ylabel": r"PxPore $V_{acc}$ (nm$^3$)",
        "png": OUT_DIR / "accessible_pore_volume_nm3.png",
        "kind": "linear",
    },
    {
        "title": "PLD",
        "pb_col": "PLD_A",
        "pp_col": "pp_PLD_A",
        "xlabel": "PoreBlazer PLD ($\\AA$)",
        "ylabel": "PxPore PLD ($\\AA$)",
        "png": OUT_DIR / "pld_A.png",
        "kind": "linear",
    },
    {
        "title": "LCD",
        "pb_col": "LCD_A",
        "pp_col": "pp_LCD_A",
        "xlabel": "PoreBlazer LCD ($\\AA$)",
        "ylabel": "PxPore LCD ($\\AA$)",
        "png": OUT_DIR / "lcd_A.png",
        "kind": "linear",
    },
    {
        "title": "LCD (global)",
        "pb_col": "LCD_A",
        "pp_col": "pp_LCD_global_A",
        "xlabel": "PoreBlazer LCD ($\\AA$)",
        "ylabel": "PxPore LCD$_{global}$ ($\\AA$)",
        "png": OUT_DIR / "lcd_global_A.png",
        "kind": "linear",
    },
    {
        "title": "LCD (global vs local)",
        "pb_col": "LCD_nm",
        "pp_col": "LCD_global_nm",
        "xlabel": "PxPore LCD (nm)",
        "ylabel": "PxPore LCD$_{global}$ (nm)",
        "png": OUT_DIR / "lcd_global_vs_local_nm.png",
        "kind": "linear",
    },
    {
        "title": "Void Fraction",
        "pb_col": "total_FV_PO",
        "pp_col": "Vvoid_frac",
        "xlabel": r"PoreBlazer $F_{V,total}$",
        "ylabel": r"PxPore $F_{V,total}$",
        "png": OUT_DIR / "void_fraction.png",
        "kind": "linear",
    },
    {
        "title": "Accessible Fraction",
        "pb_col": "net_FV_PO",
        "pp_col": "Vacc_frac",
        "xlabel": r"PoreBlazer $F_{V,acc}$",
        "ylabel": r"PxPore $F_{V,acc}$",
        "png": OUT_DIR / "accessible_fraction.png",
        "kind": "linear",
    },
]

for item in pairs:
    pb_col = item["pb_col"]
    pp_col = item["pp_col"]

    if (pb_col not in df.columns) or (pp_col not in df.columns):
        print(f"[skip] missing columns: {pb_col}, {pp_col}")
        continue

    print(f"plotting: {item['title']}")

    scatter_with_marginal(
        y=df[pb_col],
        x=df[pp_col],
        ylabel=item["xlabel"],
        xlabel=item["ylabel"],
        out_png=item["png"],
        title=item["title"],
        bins=MARGINAL_BINS,
        point_mode=POINT_MODE,
        gridsize=HEXBIN_GRIDSIZE,
        cmap=SOFT_BLUE_CMAP,
    )


# =========================================================
# 12. 时间对比图
# =========================================================
if ("pb_wall_s" in df.columns) and ("pp_wall_s" in df.columns):
    scatter_with_marginal_log(
        y=df["pb_wall_s"],
        x=df["pp_wall_s"],
        ylabel="PoreBlazer wall time (s)",
        xlabel="PxPore wall time (s)",
        out_png=OUT_DIR / "wall_time_pb_vs_pp.png",
        title="Wall Time Comparison",
        bins=MARGINAL_BINS,
        point_mode=POINT_MODE,
        gridsize=HEXBIN_GRIDSIZE,
        cmap=SOFT_BLUE_CMAP,
    )

if ("pb_cpu_s" in df.columns) and ("pp_cpu_s" in df.columns):
    scatter_with_marginal_log(
        y=df["pb_cpu_s"],
        x=df["pp_cpu_s"],
        ylabel="PoreBlazer CPU time (s)",
        xlabel="PxPore CPU time (s)",
        out_png=OUT_DIR / "cpu_time_pb_vs_pp.png",
        title="CPU Time Comparison",
        bins=MARGINAL_BINS,
        point_mode=POINT_MODE,
        gridsize=HEXBIN_GRIDSIZE,
        cmap=SOFT_BLUE_CMAP,
    )

if "speedup_wall" in df.columns:
    plot_speedup_hist(
        speedup=df["speedup_wall"],
        out_png=OUT_DIR / "speedup_hist.png",
        bins=35,
    )

if ("size_voxels" in df.columns) and ("speedup_wall" in df.columns):
    plot_speedup_vs_size(
        x=df["size_voxels"],
        y=df["speedup_wall"],
        xlabel="Voxels",
        ylabel="Wall time speedup (PB / PxPore)",
        title="Speedup vs System Size",
        out_png=OUT_DIR / "speedup_vs_voxels.png",
        bins=MARGINAL_BINS,
        point_mode=POINT_MODE,
        gridsize=HEXBIN_GRIDSIZE,
        cmap=SOFT_BLUE_CMAP,
    )

if ("size_voxels" in df.columns) and ("pb_wall_s" in df.columns) and ("pp_wall_s" in df.columns):
    plot_time_vs_size(
        pb_size=df["size_voxels"],
        pb_time=df["pb_wall_s"],
        pp_size=df["size_voxels"],
        pp_time=df["pp_wall_s"],
        xlabel="Voxels",
        out_png=OUT_DIR / "wall_time_vs_voxels.png",
        time_kind="wall",
    )

if ("size_voxels" in df.columns) and ("pb_cpu_s" in df.columns) and ("pp_cpu_s" in df.columns):
    plot_time_vs_size(
        pb_size=df["size_voxels"],
        pb_time=df["pb_cpu_s"],
        pp_size=df["size_voxels"],
        pp_time=df["pp_cpu_s"],
        xlabel="Voxels",
        out_png=OUT_DIR / "cpu_time_vs_voxels.png",
        time_kind="cpu",
    )

In [ ]:
def extract_pld_outliers(
    df,
    py_pld_col="PLD_A",
    pp_pld_col="pp_PLD_A",
    py_lcd_col="LCD_A",
    pp_lcd_col="pp_LCD_A",
    name_col="name",
    abs_diff_thresh=1.0,
    rel_diff_thresh=0.15,
    topn=100,
):
    tmp = df.copy()
    tmp = tmp[
        np.isfinite(tmp[py_pld_col]) &
        np.isfinite(tmp[pp_pld_col]) &
        np.isfinite(tmp[py_lcd_col]) &
        np.isfinite(tmp[pp_lcd_col])
    ].copy()

    tmp["pld_signed_diff_A"] = tmp[pp_pld_col] - tmp[py_pld_col]
    tmp["pld_abs_diff_A"] = np.abs(tmp["pld_signed_diff_A"])
    tmp["pld_rel_diff"] = tmp["pld_abs_diff_A"] / np.maximum(np.abs(tmp[py_pld_col]), 1e-12)
    tmp["pld_ratio_pp_over_py"] = tmp[pp_pld_col] / np.maximum(tmp[py_pld_col], 1e-12)

    flag = (
        (tmp["pld_abs_diff_A"] >= abs_diff_thresh) |
        (tmp["pld_rel_diff"] >= rel_diff_thresh)
    )
    outliers = tmp.loc[flag].copy()

    front_cols = [
        name_col,
        py_pld_col,
        pp_pld_col,
        py_lcd_col,
        pp_lcd_col,
        "pld_signed_diff_A",
        "pld_abs_diff_A",
        "pld_rel_diff",
        "pld_ratio_pp_over_py",
    ]
    remain_cols = [c for c in outliers.columns if c not in front_cols]
    outliers = outliers[front_cols + remain_cols]

    outliers = outliers.sort_values(
        ["pld_abs_diff_A", "pld_rel_diff"],
        ascending=False,
    )

    if topn is not None:
        return outliers.head(topn), outliers
    return outliers, outliers


pld_top, pld_all = extract_pld_outliers(
    df,
    py_pld_col="PLD_A",
    pp_pld_col="pp_PLD_A",
    py_lcd_col="LCD_A",
    pp_lcd_col="pp_LCD_A",
    name_col="name",
    abs_diff_thresh=1.0,
    rel_diff_thresh=0.15,
    topn=100,
)


display(pld_top)